# 04 Intervention ROI Simulation

## H4 Definition
**H4:** Targeted intervention on high-risk seller-days can reduce future severe-event harm with positive ROI, while keeping GMV-side guardrails within acceptable bounds.

## Objectives
- Rebuild the default H3 deployment ranking:
  - Logistic Regression
  - Horizon = 14 days
  - Top-K intervention policy inherited from H3
- Explicitly define the **no-intervention baseline**:
  - future severe-event count
  - future severe-event GMV
  - harm proxy in BRL
- Translate H2 customer-harm evidence into business proxies:
  - avoided incremental low ratings (monetised proxy)
  - avoided review-score loss (non-monetised KPI)
- Simulate multiple intervention scenarios under:
  - conservative
  - base
  - aggressive
  assumption profiles
- Compare:
  - captured future severe-event GMV
  - prevented future severe-event GMV
  - intervention cost
  - guardrail impact
  - net benefit and ROI
- Run explicit **K-sensitivity analysis** for top-K intervention thresholds.

## Guardrails
In this notebook, guardrail impact is defined as:
- `current_gmv_proxy_share`: share of current 14-day GMV footprint affected by intervention
- `throttled_gmv_brl`: GMV directly suppressed under restrictive actions
- `margin_loss_brl`: contribution-margin loss implied by throttled GMV

## Notes
- This is a scenario-based ROI simulation, not a causal policy estimate.
- Because H3 calibration is not yet fully validated, H4 uses a **ranked top-K intervention framework**
  rather than probability-weighted expected loss.
- Default K = 5% seller-days, inherited from H3 deployment recommendation.
- K sensitivity is evaluated at K {1%, 3%, 5%, 10%}.

## 0. Import & Settings

In [3]:
# %%
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import altair as alt
from IPython.core.interactiveshell import InteractiveShell
from pathlib import Path
import sys

pd.set_option("display.max_columns", 80)
InteractiveShell.ast_node_interactivity = "all"
alt.data_transformers.disable_max_rows()

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from config import DATA_INTERIM, DATA_PROCESSED
from data.preprocessing import load_orders_sellers

# from features.intervention_roi import (
# )

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


DataTransformerRegistry.enable('default')